# EAGF Notebook 3: RE-IoT Node-Class Fairness Analysis

This notebook analyses the RE-IoT case study from model-derived outputs:
- False-Positive-Rate Parity (FPRP) across urban / peri-urban / rural nodes
- Baseline vs. EAGF node-class disparity
- Threat type breakdown (FDIA, command injection, DoS)

All metrics below are computed from actual predictions produced in this notebook run.

[![GitHub](https://img.shields.io/badge/GitHub-aliakarma%2Feagf-blue?logo=github)](https://github.com/aliakarma/eagf)  **Repository:** [https://github.com/aliakarma/eagf](https://github.com/aliakarma/eagf)

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/aliakarma/eagf.git"
REPO_DIR_NAME = "eagf"

def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "configs").exists() and (candidate / "src").exists():
            return candidate
    return None

PROJECT_ROOT_PATH = find_project_root(Path.cwd())

if PROJECT_ROOT_PATH is None:
    clone_target = Path.cwd() / REPO_DIR_NAME
    if not clone_target.exists():
        print(f"Cloning repository into {clone_target}...")
        subprocess.run(["git", "clone", REPO_URL, str(clone_target)], check=True)
    PROJECT_ROOT_PATH = find_project_root(clone_target)
    if PROJECT_ROOT_PATH is None:
        raise RuntimeError("Could not locate project root after cloning.")
    os.chdir(PROJECT_ROOT_PATH)

PROJECT_ROOT = str(PROJECT_ROOT_PATH.resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Using PROJECT_ROOT={PROJECT_ROOT}")

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
PROJECT_ROOT = Path(PROJECT_ROOT) if 'PROJECT_ROOT' in globals() else Path.cwd()
if not (PROJECT_ROOT / 'configs').exists() and (PROJECT_ROOT.parent / 'configs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = str(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yaml

print('Imports ready.')
print(f'PROJECT_ROOT={PROJECT_ROOT}')

## 1. Generate RE-IoT Dataset

In [ ]:
import sys
if 'PROJECT_ROOT' not in globals():
    from pathlib import Path
    PROJECT_ROOT = str(Path.cwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.reiot_simulator import generate_full_reiot_dataset

dataset = generate_full_reiot_dataset(
    n_urban=20, n_periurban=20, n_rural=20,
    n_windows_per_node=50, attack_ratio=0.05, seed=42
)

print('RE-IoT Dataset Summary')
print('=' * 45)
print(f'  Train samples : {dataset["X_train"].shape[0]:,}')
print(f'  Test samples  : {dataset["X_test"].shape[0]:,}')
print(f'  Features      : {dataset["X_train"].shape[1]}')
print(f'  Attack ratio  : {dataset["y_train"].mean():.1%}')
print(f'  Node classes  : {sorted(set(dataset["groups_test"]))}')

# Class distribution in test set
print('\nTest set distribution:')
for cls in ['urban','periurban','rural']:
    mask = (dataset['groups_test'] == cls)
    n    = mask.sum()
    atk  = dataset['y_test'][mask].mean()
    print(f'  {cls:12s}: {n:4d} samples, {atk:.1%} attack rate')

## 2. Train Baseline and EAGF Detectors

In [ ]:
from src.training.eagf_trainer import train_variant

with open(os.path.join(PROJECT_ROOT, 'configs', 'reiot_default.yaml')) as f:
    config = yaml.safe_load(f)
config['training']['epochs'] = 30

print('Training Baseline and EAGF on RE-IoT data...')
m_base, base_model = train_variant(
    'baseline', config, dataset.copy(), seed=42,
    output_dir=os.path.join(PROJECT_ROOT, 'results', 'notebook_runs', 'nb3', 'baseline', 'seed_42'), return_model=True
    )
m_eagf, eagf_model = train_variant(
    'eagf', config, dataset.copy(), seed=42,
    output_dir=os.path.join(PROJECT_ROOT, 'results', 'notebook_runs', 'nb3', 'eagf', 'seed_42'), return_model=True
    )

print(f'\nBaseline: acc={m_base["accuracy"]:.3f}  FPRP={m_base["recall_parity"]:.3f}  TI={m_base["trust_index"]:.3f}')
print(f'EAGF    : acc={m_eagf["accuracy"]:.3f}  FPRP={m_eagf["recall_parity"]:.3f}  TI={m_eagf["trust_index"]:.3f}')

## 3. Node-Class FPR Breakdown

In [ ]:
node_classes = ['urban', 'periurban', 'rural']
pretty = {'urban': 'Urban', 'periurban': 'Peri-urban', 'rural': 'Rural'}

y_test = dataset['y_test']
g_test = dataset['groups_test']
y_pred_base = base_model.predict(dataset['X_test'])
y_pred_eagf = eagf_model.predict(dataset['X_test'])

def group_fpr_dr(y_true, y_pred, groups, cls):
    mask = (groups == cls)
    yt = y_true[mask]
    yp = y_pred[mask]
    neg = (yt == 0)
    pos = (yt == 1)
    fpr = float(((yp == 1) & neg).sum() / max(neg.sum(), 1))
    dr = float(((yp == 1) & pos).sum() / max(pos.sum(), 1))
    return fpr, dr

baseline_fpr, eagf_fpr = [], []
baseline_dr, eagf_dr = [], []
for cls in node_classes:
    b_fpr, b_dr = group_fpr_dr(y_test, y_pred_base, g_test, cls)
    e_fpr, e_dr = group_fpr_dr(y_test, y_pred_eagf, g_test, cls)
    baseline_fpr.append(100.0 * b_fpr)
    eagf_fpr.append(100.0 * e_fpr)
    baseline_dr.append(100.0 * b_dr)
    eagf_dr.append(100.0 * e_dr)

print('Node-Class Performance: Baseline vs. EAGF')
print('=' * 62)
print(f'{"Node Class":<14} {"Baseline FPR":>13} {"EAGF FPR":>10} {"Δ FPR":>8} {"Baseline DR":>12} {"EAGF DR":>9}')
print('-' * 62)
for cls, b_fpr, e_fpr, b_dr, e_dr in zip(node_classes, baseline_fpr, eagf_fpr, baseline_dr, eagf_dr):
    print(f'{pretty[cls]:<14} {b_fpr:>12.1f}% {e_fpr:>9.1f}% '
          f'{(e_fpr-b_fpr):>+7.1f}%  {b_dr:>10.1f}%  {e_dr:>8.1f}%')

print()
den_base = max(max(baseline_fpr), 1e-12)
den_eagf = max(max(eagf_fpr), 1e-12)
fprp_base = min(baseline_fpr) / den_base
fprp_eagf = min(eagf_fpr) / den_eagf
print(f'  Baseline FPRP (min/max) : {fprp_base:.3f}')
print(f'  EAGF     FPRP (min/max) : {fprp_eagf:.3f}')
print(f'  Improvement             : {fprp_eagf - fprp_base:+.3f} ({(fprp_eagf-fprp_base)*100:+.1f} pp)')
if den_eagf <= 1e-12:
    print('  Note: EAGF FPR is zero across all groups; parity ratio is numerically stabilized.')

## 4. Figure: Node-Class FPR Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: FPR comparison
x = np.arange(len(node_classes))
w = 0.35
ax = axes[0]
bars_b = ax.bar(x - w/2, baseline_fpr, w, label='Baseline (M0)', color='#F08080', edgecolor='white')
bars_e = ax.bar(x + w/2, eagf_fpr,     w, label='EAGF (M5)',     color='#3CB371', edgecolor='white')

for bar in list(bars_b) + list(bars_e):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels([pretty[c] for c in node_classes])
ax.set_ylabel('False Positive Rate (%)')
ax.set_title('FPR by Node Class: Baseline vs. EAGF\n(lower = fewer false alarms)', fontsize=10)
ax.legend(); ax.grid(axis='y', alpha=0.2)
ax.spines[['top','right']].set_visible(False)

# Right: Detection Rate
ax2 = axes[1]
bars_b2 = ax2.bar(x - w/2, baseline_dr, w, label='Baseline (M0)', color='#F08080', edgecolor='white')
bars_e2 = ax2.bar(x + w/2, eagf_dr,     w, label='EAGF (M5)',     color='#3CB371', edgecolor='white')

for bar in list(bars_b2) + list(bars_e2):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

ax2.set_xticks(x); ax2.set_xticklabels([pretty[c] for c in node_classes])
ax2.set_ylabel('Detection Rate (%)')
ax2.set_ylim(0, 100)
ax2.set_title('Detection Rate by Node Class\n(higher = fewer missed attacks)', fontsize=10)
ax2.legend(); ax2.grid(axis='y', alpha=0.2)
ax2.spines[['top','right']].set_visible(False)

plt.suptitle('RE-IoT Node-Class Fairness: Baseline vs. EAGF (model-derived)', fontsize=12, y=1.02)
plt.tight_layout()
out = os.path.join(PROJECT_ROOT, 'figures', 'notebook3_reiot_fairness.png')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print(f'Figure saved → {out}')

## 5. Operational Impact Estimation

In [ ]:
# Estimate cost impact from measured rural false-positive reduction

n_rural_nodes       = 40
windows_per_day     = 1440  # 1-Hz sampling, 60-s windows
cost_per_inspection = 500   # USD mid-estimate

rural_idx = node_classes.index('rural')
fpr_reduction = max((baseline_fpr[rural_idx] - eagf_fpr[rural_idx]) / 100.0, 0.0)
daily_inspections_avoided = n_rural_nodes * windows_per_day * fpr_reduction
daily_cost_saved = daily_inspections_avoided * cost_per_inspection

print('Operational Impact Estimate (Rural Nodes, 40-Node Deployment)')
print('=' * 60)
print(f'  Rural FPR reduction     : {fpr_reduction*100:.2f} pp')
print(f'  Daily windows/node      : {windows_per_day:,}')
print(f'  False alarms avoided/day: {daily_inspections_avoided:,.0f}')
print(f'  Cost per inspection     : ${cost_per_inspection:,}')
print(f'  Estimated daily saving  : ${daily_cost_saved:,.0f}')
print(f'  Estimated annual saving : ${daily_cost_saved * 365:,.0f}')